# Other Utilities — Reference

| Utility | Purpose |
|---|---|
| `.pt.cols()` | List column names sorted or in original order |
| `.pt.glimpse()` | Transposed column overview with dtypes and sample values (returns self for chaining) |
| `.pt.handle_missing()` | Fill string NaN with a marker, numeric NaN with 0, strip whitespace |
| `.pt.clean_columns()` | Clean column header names (strip, squeeze, fill, case, dedupe) |
| `.pt.replace_values()` | Replace values, optionally scoped to columns, exact or substring match |
| `df.to_clip()` | Copy DataFrame to clipboard (tab-separated, no index) |

---

In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src'))
if not os.path.exists(_src): _src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import pytae as pt

penguins = pt.sample_data['penguins']
titanic = pt.sample_data['titanic']

## `cols()` — list column names

In [2]:
# Alphabetical column list — useful for quick discovery
penguins.pt.cols()

['bill_depth_mm',
 'bill_length_mm',
 'body_mass_g',
 'flipper_length_mm',
 'island',
 'sex',
 'species']

In [3]:
# Descending alphabetical sort — useful when column names form a natural reverse order
penguins.pt.cols(ascending=False)

['species',
 'sex',
 'island',
 'flipper_length_mm',
 'body_mass_g',
 'bill_length_mm',
 'bill_depth_mm']

## `glimpse()` — transposed column overview

`.pt.glimpse()` prints a compact, transposed overview of the DataFrame inspired by R/dplyr's `glimpse()`:
- Summarizes row count, column count, and in-memory footprint.
- Lists every column name alongside its dtype and inline sample values.
- Non-destructive: prints to stdout and returns `self` (the unchanged DataFrame) so it can be inserted anywhere inside a method chain for debugging.

In [4]:
# Direct glimpse overview
penguins.pt.glimpse()

Rows: 344
Columns: 7
$ species           <str>     'Adelie', 'Adelie', 'Adelie', 'Adelie', 'Adelie...
$ island            <str>     'Torgersen', 'Torgersen', 'Torgersen', 'Torgers...
$ bill_length_mm    <float64> 39.1, 39.5, 40.3, NA, 36.7, 39.3, 38.9, 39.2, 3...
$ bill_depth_mm     <float64> 18.7, 17.4, 18.0, NA, 19.3, 20.6, 17.8, 19.6, 1...
$ flipper_length_mm <float64> 181.0, 186.0, 195.0, NA, 193.0, 190.0, 181.0, 1...
$ body_mass_g       <float64> 3750.0, 3800.0, 3250.0, NA, 3450.0, 3650.0, 362...
$ sex               <str>     'Male', 'Female', 'Female', NA, 'Female', 'Male...


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,NaN,NaN,NaN,NaN,NaN
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female


In [5]:
# Non-destructive: returns self, allowing inspection mid-pipeline
(
    penguins
    .pt.filter("body_mass_g > 4000")
    .pt.glimpse()
    .pt.select("species", "island", "body_mass_g")
    .head(3)
)

Rows: 172
Columns: 7
$ species           <str>     'Adelie', 'Adelie', 'Adelie', 'Adelie', 'Adelie...
$ island            <str>     'Torgersen', 'Torgersen', 'Torgersen', 'Torgers...
$ bill_length_mm    <float64> 39.2, 42.0, 34.6, 42.5, 46.0, 39.2, 39.8, 44.1...
$ bill_depth_mm     <float64> 19.6, 20.2, 21.1, 20.7, 21.5, 21.1, 19.1, 19.7...
$ flipper_length_mm <float64> 195.0, 190.0, 198.0, 197.0, 194.0, 196.0, 184.0...
$ body_mass_g       <float64> 4675.0, 4250.0, 4400.0, 4500.0, 4200.0, 4150.0...
$ sex               <str>     'Male', NA, 'Male', 'Male', 'Male', 'Male', 'Ma...


,species,island,body_mass_g
7,Adelie,Torgersen,4675.0
9,Adelie,Torgersen,4250.0
14,Adelie,Torgersen,4400.0


## `handle_missing()` — clean before aggregating
Fills string/category NaN with a visible marker (default `.`) and numeric NaN with `0`. Also strips leading/trailing whitespace from string columns.

In [6]:
# Default fillna='.': NaN sex becomes '.', NaN numerics become 0
# Then aggregate to see the '.' group alongside real groups
(penguins
 .pt.handle_missing()
)


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,0.0,0.0,0.0,0.0,.
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,0.0,0.0,0.0,0.0,.
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female


In [7]:
# Custom fillna marker — makes missing group explicit in reports
( penguins
 .pt.handle_missing(fillna='Unknown')
)


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,0.0,0.0,0.0,0.0,Unknown
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,0.0,0.0,0.0,0.0,Unknown
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female


In [8]:
# Use a single-char marker like '$' to make missing data stand out in reports
( penguins
 .pt.handle_missing(fillna='$')
)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,0.0,0.0,0.0,0.0,$
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,0.0,0.0,0.0,0.0,$
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female


## `df.to_clip()` — copy to clipboard
`df.to_clip()` copies the DataFrame to the system clipboard as tab-separated text (no index). Paste directly into Excel, Google Sheets, or any text editor. Invisible in notebook output — run it and then paste.

In [9]:
# df.to_clip() copies the current DataFrame to clipboard — paste into Excel/Sheets
# Run this cell then Cmd+V in any spreadsheet application
(penguins
 .pt.select('species', 'island', 'bill_length_mm', 'body_mass_g')
 .pt.by('species', 'island')
 .pt.agg(a=['mean', 'n'])
).to_clip()


## `clean_columns()` — clean messy header names
Cleans column header names in a fixed order: `strip` -> `strip_special` -> `squeeze` -> `fill` -> `case` -> `dedupe`. Handy after loading a spreadsheet export with inconsistent spacing/casing.

In [10]:
# messy headers: leading/trailing spaces, repeated internal spaces, mixed case
messy = penguins.rename(columns={
    'species': '  Species  ',
    'island': 'Island   Name',
    'body_mass_g': 'Body Mass (g)',
})

(messy
 .pt.clean_columns(strip=True, squeeze=True, strip_special=True, fill='_', case='lower')
 .pt.cols(ascending=None)
)

['species',
 'island_name',
 'bill_length_mm',
 'bill_depth_mm',
 'flipper_length_mm',
 'body_mass_g',
 'sex']

In [11]:
# Method chain: clean names, then display them in original column order
(
    penguins
    .pt.clean_columns(case='proper')
    .pt.cols(ascending=None)
)


['Species',
 'Island',
 'Bill_Length_Mm',
 'Bill_Depth_Mm',
 'Flipper_Length_Mm',
 'Body_Mass_G',
 'Sex']

## `replace_values()` — replace values, whole-cell or substring
Wraps pandas `replace()`. `exact=True` (default) only replaces a cell whose entire value matches a key exactly; `exact=False` replaces a key anywhere it occurs as a substring, leaving the rest of the cell untouched. Optionally scope to specific columns with `c=`. Parameter names (`v=`, `c=`, `exact=`) match the `-replace_values` CLI flag.

In [12]:
# exact=True (default): only cells that equal 'Adelie'/'Male' exactly are replaced
(penguins
 .pt.replace_values({'Adelie': 'Adelie (renamed)', 'Male': 'M'})
 .pt.select('species', 'sex')
 .drop_duplicates()
)

,species,sex
0,Adelie (renamed),M
1,Adelie (renamed),Female
3,Adelie (renamed),NaN
152,Chinstrap,Female
153,Chinstrap,M
220,Gentoo,Female
221,Gentoo,M
246,Gentoo,NaN


In [13]:
# c= restricts the replace to specific columns — 'island' untouched even though
# it never contained 'Male' anyway, this shows the scoping mechanism itself
(penguins
 .pt.replace_values({'Male': 'M', 'Female': 'F'}, c='sex')
 .pt.select('island', 'sex')
 .drop_duplicates()
)

,island,sex
0,Torgersen,M
1,Torgersen,F
3,Torgersen,NaN
20,Biscoe,F
21,Biscoe,M
30,Dream,F
31,Dream,M
47,Dream,NaN
246,Biscoe,NaN


In [14]:
# exact=False: substring match anywhere in the cell, rest of the cell untouched
import pandas as pd

notes = pd.DataFrame({'note': ['a magician performed', 'not a magician exactly', 'no match here']})
notes.pt.replace_values({'a magician': 'the magic'}, exact=False)

,note
0,the magic performed
1,not the magic exactly
2,no match here
